In [23]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. SETTINGS
# ============================================================

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# We use a dictionary consistently throughout this example.
config = {
    "embedding_size": 8,
    "num_heads": 2,
    "context_length": 16,
}

print("Device:", device)


# ============================================================
# 2. TOKENIZATION: CHARACTERS <-> INTEGER IDs
# ============================================================

text = "hello world"

characters = sorted(set(text))
vocab_size = len(characters)

char_to_id = {
    character: index
    for index, character in enumerate(characters)
}

id_to_char = {
    index: character
    for index, character in enumerate(characters)
}


def encode(string):
    return [char_to_id[character] for character in string]


def decode(token_ids):
    return "".join(id_to_char[int(index)] for index in token_ids)


print("\nVocabulary:", char_to_id)


# ============================================================
# 3. PREPARE ONE INPUT EXAMPLE
# ============================================================

input_text = "hell"

# The outer list adds a batch dimension.
# Shape: [batch size, sequence length] = [1, 4]
input_ids = torch.tensor(
    [encode(input_text)],
    dtype=torch.long,
    device=device,
)

batch_size, sequence_length = input_ids.shape

assert sequence_length <= config["context_length"]

print("\nInput text:", input_text)
print("Input IDs:", input_ids)
print("Input shape:", input_ids.shape)


# ============================================================
# 4. TOKEN AND POSITION EMBEDDINGS
# ============================================================

# One trainable vector for each character.
token_embedding = nn.Embedding(
    vocab_size,
    config["embedding_size"],
).to(device)

# One trainable vector for each possible position.
position_embedding = nn.Embedding(
    config["context_length"],
    config["embedding_size"],
).to(device)

# Positions: [0, 1, 2, 3]
positions = torch.arange(
    sequence_length,
    device=device,
)

# Shape: [1, 4, 8]
token_vectors = token_embedding(input_ids)

# Shape: [4, 8]
position_vectors = position_embedding(positions)

# Position vectors are added to every example in the batch.
# Shape: [1, 4, 8]
hidden = token_vectors + position_vectors

print("\nToken vectors shape:", token_vectors.shape)
print("Position vectors shape:", position_vectors.shape)
print("Combined vectors shape:", hidden.shape)


# ============================================================
# 5. DEFINE CAUSAL MULTI-HEAD SELF-ATTENTION
# ============================================================

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        embedding_size = config["embedding_size"]
        num_heads = config["num_heads"]
        context_length = config["context_length"]

        if embedding_size % num_heads != 0:
            raise ValueError(
                "embedding_size must be divisible by num_heads"
            )

        self.num_heads = num_heads
        self.head_size = embedding_size // num_heads

        # Compute queries, keys, and values together.
        # With embedding_size=8, this produces 24 numbers.
        self.qkv = nn.Linear(
            embedding_size,
            3 * embedding_size,
        )

        # Mix the features after joining the attention heads.
        self.output_projection = nn.Linear(
            embedding_size,
            embedding_size,
        )

        # True means the position is visible.
        # Lower-triangular mask blocks future positions.
        mask = torch.tril(
            torch.ones(
                context_length,
                context_length,
                dtype=torch.bool,
            )
        )

        # A buffer moves to CPU/GPU with the layer,
        # but is not a trainable parameter.
        self.register_buffer("causal_mask", mask)

    def forward(self, x):
        # B: batch size
        # T: number of tokens
        # C: embedding size
        B, T, C = x.shape

        if T > self.causal_mask.shape[0]:
            raise ValueError("Input exceeds context_length")

        # ----------------------------------------------------
        # A. Create queries, keys, and values
        # ----------------------------------------------------

        # [B, T, C] -> [B, T, 3*C]
        combined_qkv = self.qkv(x)

        # Each tensor has shape [B, T, C].
        q, k, v = combined_qkv.chunk(3, dim=-1)

        # ----------------------------------------------------
        # B. Split features into attention heads
        # ----------------------------------------------------

        # [B, T, C] -> [B, H, T, D]
        # H = num_heads; D = head_size
        q = q.reshape(
            B, T, self.num_heads, self.head_size
        ).transpose(1, 2)

        k = k.reshape(
            B, T, self.num_heads, self.head_size
        ).transpose(1, 2)

        v = v.reshape(
            B, T, self.num_heads, self.head_size
        ).transpose(1, 2)

        # ----------------------------------------------------
        # C. Compare queries with keys
        # ----------------------------------------------------

        # [B, H, T, D] @ [B, H, D, T]
        # Result: [B, H, T, T]
        scores = q @ k.transpose(-2, -1)

        scores = scores / math.sqrt(self.head_size)

        # ----------------------------------------------------
        # D. Block access to future positions
        # ----------------------------------------------------

        allowed_positions = self.causal_mask[:T, :T]

        scores = scores.masked_fill(
            ~allowed_positions,
            float("-inf"),
        )

        # ----------------------------------------------------
        # E. Convert scores into attention weights
        # ----------------------------------------------------

        # Each row sums to 1.
        # Masked positions receive weight 0.
        attention_weights = F.softmax(scores, dim=-1)

        # ----------------------------------------------------
        # F. Mix the value vectors
        # ----------------------------------------------------

        # [B, H, T, T] @ [B, H, T, D]
        # Result: [B, H, T, D]
        mixed = attention_weights @ v

        # ----------------------------------------------------
        # G. Join heads and apply the output projection
        # ----------------------------------------------------

        # [B, H, T, D] -> [B, T, H, D] -> [B, T, C]
        mixed = (
            mixed.transpose(1, 2)
            .contiguous()
            .reshape(B, T, C)
        )

        output = self.output_projection(mixed)

        # Return weights too so we can inspect them.
        return output, attention_weights


# ============================================================
# 6. CREATE AND RUN THE ATTENTION LAYER
# ============================================================

attention = CausalSelfAttention(config).to(device)

attention_output, attention_weights = attention(hidden)

print("\nAttention output shape:", attention_output.shape)
print("Attention weights shape:", attention_weights.shape)

print("\nAttention output:")
print(attention_output.detach().cpu())


# ============================================================
# 7. INSPECT THE ATTENTION WEIGHTS
# ============================================================

torch.set_printoptions(precision=3, sci_mode=False)

for head_index in range(config["num_heads"]):
    print(f"\nHead {head_index + 1} weights:")

    # First batch example, selected head.
    print(
        attention_weights[0, head_index]
        .detach()
        .cpu()
    )

print("\nRow sums — each should be approximately 1:")
print(attention_weights.sum(dim=-1).detach().cpu())


# ============================================================
# 8. CHECK THAT FUTURE POSITIONS GET ZERO WEIGHT
# ============================================================

future_weights = torch.triu(
    attention_weights,
    diagonal=1,
)

assert torch.all(future_weights == 0).item()

print("\nPassed: future positions receive zero attention weight.")

Device: cuda

Vocabulary: {' ': 0, 'd': 1, 'e': 2, 'h': 3, 'l': 4, 'o': 5, 'r': 6, 'w': 7}

Input text: hell
Input IDs: tensor([[3, 2, 4, 4]], device='cuda:0')
Input shape: torch.Size([1, 4])

Token vectors shape: torch.Size([1, 4, 8])
Position vectors shape: torch.Size([4, 8])
Combined vectors shape: torch.Size([1, 4, 8])

Attention output shape: torch.Size([1, 4, 8])
Attention weights shape: torch.Size([1, 2, 4, 4])

Attention output:
tensor([[[ 0.2183,  0.6880, -0.6097,  0.0856,  0.7736, -0.4349, -0.8276,
           0.2697],
         [ 0.4402,  0.3951, -0.5171, -0.0530,  0.3143, -0.4217, -0.4730,
           0.2143],
         [ 0.3924,  0.3942, -0.4445,  0.0482,  0.4009, -0.5193, -0.2608,
          -0.0080],
         [ 0.3985,  0.1661, -0.2243,  0.1615, -0.0342, -0.3927,  0.0063,
          -0.1011]]])

Head 1 weights:
tensor([[1.000, 0.000, 0.000, 0.000],
        [0.509, 0.491, 0.000, 0.000],
        [0.174, 0.317, 0.509, 0.000],
        [0.277, 0.218, 0.246, 0.260]])

Head 2 weights

In [24]:
# ============================================================
# COMPLETE THE GPT PROJECT
# Run the previous big cell first.
# ============================================================

import csv
import math
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

if "CausalSelfAttention" not in globals():
    raise RuntimeError(
        "Run the previous cell defining CausalSelfAttention first."
    )


# ============================================================
# 1. SETTINGS
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# True: short test of the whole project.
# False: longer training, usually better generated text.
QUICK_RUN = True
RUN_EXPERIMENTS = True

config = {
    "embedding_size": 64,
    "num_heads": 4,
    "num_layers": 2,
    "context_length": 32,
    "batch_size": 8,
    "learning_rate": 0.001,
    "steps": 30 if QUICK_RUN else 600,
    "eval_every": 15 if QUICK_RUN else 100,
    "eval_batches": 3 if QUICK_RUN else 10,
}

output_dir = Path("gpt_outputs")
output_dir.mkdir(exist_ok=True)

print("Device:", device)
print("Output folder:", output_dir.resolve())


# ============================================================
# 2. PREPARE A PRACTICE CORPUS
# ============================================================

subjects = [
    "the reader",
    "the teacher",
    "the student",
    "the writer",
]

actions = [
    "opens a book",
    "writes a story",
    "reads a poem",
    "asks a question",
]

places = [
    "in the library",
    "by the window",
    "in the morning",
    "after the rain",
]

sentences = []

for subject in subjects:
    for action in actions:
        for place in places:
            sentences.append(f"{subject} {action} {place}.\n")

text = "".join(sentences) * 10

# OPTIONAL:
# To train on your own UTF-8 text file instead, uncomment:
# text = Path("input.txt").read_text(encoding="utf-8")

print("\nNumber of characters:", len(text))
print(text[:200])


# ============================================================
# 3. REBUILD THE TOKENIZER FOR THIS CORPUS
# ============================================================

characters = sorted(set(text))
vocab_size = len(characters)

char_to_id = {
    character: index
    for index, character in enumerate(characters)
}

id_to_char = {
    index: character
    for index, character in enumerate(characters)
}


def encode(string):
    unknown = set(string) - set(char_to_id)

    if unknown:
        raise ValueError(
            f"Characters not in the vocabulary: {sorted(unknown)!r}"
        )

    return [char_to_id[character] for character in string]


def decode(ids):
    return "".join(id_to_char[int(index)] for index in ids)


config["vocab_size"] = vocab_size

print("Vocabulary size:", vocab_size)


# ============================================================
# 4. SPLIT DATA BEFORE CREATING TRAINING WINDOWS
# ============================================================

data = torch.tensor(encode(text), dtype=torch.long)

split_position = int(0.9 * len(data))

train_data = data[:split_position]
val_data = data[split_position:]

if min(len(train_data), len(val_data)) <= config["context_length"]:
    raise ValueError(
        "Your text is too short. Add more text or reduce context_length."
    )

# The repeated practice corpus contains similar/repeated examples
# in both splits. Validation here checks learning of these patterns,
# not generalization to substantially different text.


def get_batch(split, cfg, generator=None):
    source = train_data if split == "train" else val_data
    T = cfg["context_length"]

    # Pick random starting positions.
    starts = torch.randint(
        len(source) - T,
        (cfg["batch_size"],),
        generator=generator,
    )

    # Inputs and targets are shifted by one character.
    x = torch.stack([
        source[start:start + T]
        for start in starts.tolist()
    ])

    y = torch.stack([
        source[start + 1:start + T + 1]
        for start in starts.tolist()
    ])

    return x.to(device), y.to(device)


# ============================================================
# 5. DEFINE A TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        C = cfg["embedding_size"]

        self.norm1 = nn.LayerNorm(C)
        self.attention = CausalSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(C)

        # Transforms each position's features separately.
        self.feed_forward = nn.Sequential(
            nn.Linear(C, 4 * C),
            nn.GELU(),
            nn.Linear(4 * C, C),
        )

    def forward(self, x):
        # Our previous attention class returns two items:
        # output vectors and attention weights.
        attention_output, _ = self.attention(self.norm1(x))

        # Residual connection: retain x and add an update.
        x = x + attention_output

        # Another residual connection around the feed-forward network.
        x = x + self.feed_forward(self.norm2(x))

        return x


# ============================================================
# 6. ASSEMBLE THE COMPLETE GPT
# ============================================================

class MiniGPT(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.config = cfg.copy()

        C = cfg["embedding_size"]
        V = cfg["vocab_size"]
        T = cfg["context_length"]

        self.token_embedding = nn.Embedding(V, C)
        self.position_embedding = nn.Embedding(T, C)

        self.blocks = nn.ModuleList([
            TransformerBlock(cfg)
            for _ in range(cfg["num_layers"])
        ])

        self.final_norm = nn.LayerNorm(C)

        # Produce one score for every possible next character.
        self.output_layer = nn.Linear(C, V)

    def forward(self, ids, targets=None):
        B, T = ids.shape

        if T > self.config["context_length"]:
            raise ValueError("Input is longer than context_length.")

        positions = torch.arange(T, device=ids.device)

        # [B, T] -> [B, T, C]
        x = (
            self.token_embedding(ids)
            + self.position_embedding(positions)
        )

        for block in self.blocks:
            x = block(x)

        # [B, T, C] -> [B, T, V]
        logits = self.output_layer(self.final_norm(x))

        loss = None

        if targets is not None:
            # Each position is a separate next-character prediction.
            loss = F.cross_entropy(
                logits.reshape(-1, self.config["vocab_size"]),
                targets.reshape(-1),
            )

        return logits, loss


# ============================================================
# 7. VALIDATION: MEASURE LOSS WITHOUT UPDATING WEIGHTS
# ============================================================

@torch.no_grad()
def evaluate(model, cfg):
    was_training = model.training
    model.eval()

    metrics = {}

    for split in ("train", "val"):
        # Fixed evaluation windows make comparisons more consistent.
        generator = torch.Generator().manual_seed(2026)

        losses = []

        for _ in range(cfg["eval_batches"]):
            x, y = get_batch(split, cfg, generator)
            _, loss = model(x, y)
            losses.append(loss.item())

        metrics[split] = sum(losses) / len(losses)

    model.train(was_training)

    return metrics


# ============================================================
# 8. TRAINING LOOP
# ============================================================

def train_model(cfg):
    # Each experiment starts with fresh weights.
    torch.manual_seed(42)

    model = MiniGPT(cfg).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["learning_rate"],
    )

    # Separate generator keeps training batches consistent.
    batch_generator = torch.Generator().manual_seed(42)

    history = []
    best_val_loss = float("inf")
    best_state = None

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for step in range(cfg["steps"] + 1):

        # Evaluate initially, periodically, and after the final update.
        if step % cfg["eval_every"] == 0 or step == cfg["steps"]:
            metrics = evaluate(model, cfg)

            history.append({
                "step": step,
                "train_loss": metrics["train"],
                "val_loss": metrics["val"],
            })

            print(
                f"Step {step:4d} | "
                f"Train: {metrics['train']:.3f} | "
                f"Validation: {metrics['val']:.3f}"
            )

            if metrics["val"] < best_val_loss:
                best_val_loss = metrics["val"]

                # Copy weights: otherwise later training would change them.
                best_state = {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in model.state_dict().items()
                }

        if step == cfg["steps"]:
            break

        model.train()

        # A. Get input examples and correct answers.
        x, y = get_batch("train", cfg, batch_generator)

        # B. Predict and measure error.
        _, loss = model(x, y)

        # C. Clear gradients from the previous update.
        optimizer.zero_grad(set_to_none=True)

        # D. Calculate gradients.
        loss.backward()

        # E. Limit unusually large gradients.
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # F. Update the trainable parameters.
        optimizer.step()

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start_time

    # Restore the best measured validation checkpoint.
    model.load_state_dict(best_state)
    model.eval()

    return model, history, elapsed


# ============================================================
# 9. TRAIN THE BASELINE MODEL
# ============================================================

print("\nTRAINING BASELINE")

model, history, elapsed = train_model(config)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("\nParameters:", parameter_count)
print(f"Training plus evaluation time: {elapsed:.1f} seconds")


# ============================================================
# 10. GENERATE TEXT
# ============================================================

@torch.no_grad()
def generate(model, prompt, new_characters=150, temperature=0.8):
    if not prompt:
        raise ValueError("Provide a nonempty prompt.")

    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")

    was_training = model.training
    model.eval()

    model_device = next(model.parameters()).device

    ids = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=model_device,
    )

    for _ in range(new_characters):
        # Use only the most recent context window.
        context = ids[:, -model.config["context_length"]:]

        logits, _ = model(context)

        # Last position predicts the character following the input.
        next_logits = logits[:, -1, :] / temperature

        probabilities = F.softmax(next_logits, dim=-1)

        next_id = torch.multinomial(
            probabilities,
            num_samples=1,
        )

        ids = torch.cat((ids, next_id), dim=1)

    model.train(was_training)

    return decode(ids[0].tolist())


# This prompt is guaranteed to use known characters.
prompt = text[:12]

torch.manual_seed(123)

generated_text = generate(model, prompt)

print("\nGENERATED TEXT")
print(generated_text)


# ============================================================
# 11. COMPARE HYPERPARAMETERS
# ============================================================

def make_result(name, cfg, trained_model, history, seconds):
    best = min(history, key=lambda row: row["val_loss"])

    return {
        "experiment": name,
        "embedding_size": cfg["embedding_size"],
        "heads": cfg["num_heads"],
        "layers": cfg["num_layers"],
        "parameters": sum(p.numel() for p in trained_model.parameters()),
        "best_step": best["step"],
        "validation_loss": best["val_loss"],
        "seconds": seconds,
    }


results = [
    make_result("baseline", config, model, history, elapsed)
]

if RUN_EXPERIMENTS:
    variants = [
        ("fewer_heads", {"num_heads": 2}),
        ("more_heads", {"num_heads": 8}),
        ("more_layers", {"num_layers": 3}),
        ("wider_embeddings", {"embedding_size": 128}),
    ]

    for name, changes in variants:
        print(f"\nEXPERIMENT: {name}")

        # Copy baseline settings, then change only the selected setting.
        experiment_config = config.copy()
        experiment_config.update(changes)

        experiment_model, experiment_history, seconds = train_model(
            experiment_config
        )

        results.append(
            make_result(
                name,
                experiment_config,
                experiment_model,
                experiment_history,
                seconds,
            )
        )

        torch.manual_seed(123)
        print(generate(experiment_model, prompt, new_characters=80))

        # Keep the baseline model; release each experiment model.
        del experiment_model

        if device.type == "cuda":
            torch.cuda.empty_cache()


print("\nFINAL COMPARISON")

for result in results:
    print(
        f"{result['experiment']:18s} | "
        f"C={result['embedding_size']:3d} | "
        f"H={result['heads']} | "
        f"L={result['layers']} | "
        f"Parameters={result['parameters']:7d} | "
        f"Val loss={result['validation_loss']:.3f}"
    )


# ============================================================
# 12. SAVE RESULTS AND BASELINE MODEL
# ============================================================

with (output_dir / "experiments.csv").open(
    "w", newline="", encoding="utf-8"
) as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)

(output_dir / "generated_text.txt").write_text(
    generated_text,
    encoding="utf-8",
)

checkpoint_path = output_dir / "mini_gpt.pt"

torch.save(
    {
        "config": config,
        "characters": characters,
        "model_state": {
            name: tensor.detach().cpu()
            for name, tensor in model.state_dict().items()
        },
    },
    checkpoint_path,
)


# ============================================================
# 13. RELOAD THE MODEL AND VERIFY ITS PREDICTIONS
# ============================================================

# Load only a checkpoint you trust.
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True,
)

# Restore the exact vocabulary used during training.
characters = checkpoint["characters"]

char_to_id = {
    character: index
    for index, character in enumerate(characters)
}

id_to_char = {
    index: character
    for index, character in enumerate(characters)
}

loaded_model = MiniGPT(checkpoint["config"]).to(device)
loaded_model.load_state_dict(checkpoint["model_state"])
loaded_model.eval()

test_ids = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device,
)

model.eval()

with torch.no_grad():
    original_logits, _ = model(test_ids)
    loaded_logits, _ = loaded_model(test_ids)

    assert torch.allclose(
        original_logits,
        loaded_logits,
        atol=1e-5,
        rtol=1e-5,
    )

print("\nReload verification passed.")

print("\nTEXT FROM RELOADED MODEL")
print(generate(loaded_model, prompt, new_characters=100))

print("\nProject complete.")
print("Saved files:", output_dir.resolve())

Device: cuda
Output folder: C:\Users\user\VSCodeFiles\my_GPT\gpt_outputs

Number of characters: 26560
the reader opens a book in the library.
the reader opens a book by the window.
the reader opens a book in the morning.
the reader opens a book after the rain.
the reader writes a story in the library.
Vocabulary size: 25

TRAINING BASELINE
Step    0 | Train: 3.365 | Validation: 3.356
Step   15 | Train: 2.440 | Validation: 2.430
Step   30 | Train: 2.009 | Validation: 1.977

Parameters: 105369
Training plus evaluation time: 0.5 seconds

GENERATED TEXT
the reader oopen a then deesu sthe w wftsn.
tfthe rer.o ln sacsgar thdes np frrr in de thrdrin.
tin.
therde brilr te aee artheae the lc inhe 
the.
ter t srehedrn 

EXPERIMENT: fewer_heads
Step    0 | Train: 3.366 | Validation: 3.357
Step   15 | Train: 2.439 | Validation: 2.429
Step   30 | Train: 2.007 | Validation: 1.974
the reader oopen a then deesu sthe w wftsn.
tfthe rer.o ln sa sgar thdes np frrr in de thrdr

EXPERIMENT: more_heads
Step